<a href="https://colab.research.google.com/github/pop756/Quantum_KAN/blob/EMT/Real_backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!git clone -b EMT https://github.com/pop756/Quantum_KAN.git
%cd Quantum_KAN
!pip install -r requirements.txt

Cloning into 'Quantum_KAN'...
remote: Enumerating objects: 894, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 894 (delta 15), reused 22 (delta 1), pack-reused 837
Receiving objects: 100% (894/894), 21.61 MiB | 10.10 MiB/s, done.
Resolving deltas: 100% (133/133), done.
/content/Quantum_KAN
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 11.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 15.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 15.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 9.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.9/249.9 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.8/191.8 kB 11.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import os
import pandas as pd
import re

# Initialize an empty dictionary to store the results
X_dict = {}

# Specify the directory to search for CSV files
directory = "ibm_osaka"

# Get a list of all CSV files in the specified directory
csv_files = [
    file
    for file in os.listdir(directory)
    if file.endswith(".csv") and file.startswith("ecr_error_")
]

# Regular expression to extract (q_index1, q_index2) from the file name
pattern = re.compile(r"ecr_error_\d+_\((\d+),\s*(\d+)\)_y_pulse.csv")

for file in csv_files:
    match = pattern.match(file)
    if match:
        q_index1 = int(match.group(1))
        q_index2 = int(match.group(2))

        # Read the CSV file into a DataFrame
        df = pd.read_csv(os.path.join(directory, file))

        # Filter rows where the index is greater than 5
        filtered_df = df[df.index > 5]

        # Get the row with the minimum value in ['x0']
        min_x0_row = filtered_df.loc[
            (abs(filtered_df["x1"]) + abs(filtered_df["x0"])).idxmin()
        ]
        print((abs(filtered_df["x1"]) + abs(filtered_df["x0"])).min())
        # Get the ['xval'] value from the row with the minimum ['x0']
        xval = min_x0_row["xval"]

        # Store the result in the dictionary
        X_dict[(q_index1, q_index2)] = xval

0.017999999999999898
0.1164999999999998
0.004
0.0035
0.0114999999999998
0.0144999999999999
0.0289999999999999


In [2]:
import qiskit
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
import numpy as np
from qiskit.compiler import transpile
from qiskit_ibm_runtime.options import TwirlingOptions
from qiskit.quantum_info import SparsePauliOp
import uuid
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
def create_trotterized_ising_model_circuit(num_qubits, trotter_steps, time):
    """
    Trotterization을 사용하여 Ising 모델의 양자 서킷을 생성합니다.
    각 Rz 게이트의 파라미터는 Qiskit Parameter로 동적으로 설정됩니다.

    Parameters:
    - num_qubits: 큐비트의 수
    - trotter_steps: Trotterization 단계 수
    - time: 진화 시간

    Returns:
    - QuantumCircuit: 생성된 양자 서킷
    - dict: 서킷 파라미터 사전
    """
    qc = QuantumCircuit(num_qubits)
    delta_t = 1

    # 파라미터 설정
    h_params = [Parameter(f'h_{i}') for i in range(num_qubits)]
    J_params = [[Parameter(f'J_{i}_{j}') for j in range(num_qubits)] for i in range(num_qubits)]

    for _ in range(trotter_steps):
        # 자기 상호작용 항목 추가
        
        for i in range(num_qubits):
            qc.rx(2 * h_params[i] * delta_t, i)

        # 스핀 간 상호작용 항목 추가
        for i in range(num_qubits-1):
            if J_params[i][i+1] != 0:
                qc.cx(i, i+1)
                qc.rz(2 * J_params[i][i+1] * delta_t, i+1)
                qc.cx(i, i+1)

    return qc, h_params, J_params


# ECR 게이트를 제거하는 함수
def remove_ecr_gates(circuit):
    new_circuit = QuantumCircuit(circuit.num_qubits)
    for instr, qargs, cargs in circuit.data:
        if instr.name != 'ecr':
            new_circuit.append(instr, qargs, cargs)
    return new_circuit

# 파라미터 설정
trotter_steps = 3  # Trotterization 단계 수
time = 1.0  # 진화 시간
num_qubits = 8
init_p  =['I' for i in range(num_qubits)]
p_list = []
for i in range(num_qubits):
    temp = init_p.copy()
    temp[i] = 'Z'
    p_list.append(''.join(temp))


hamiltonian = SparsePauliOp.from_list([(p_list[i],1) for i in range(num_qubits)])


In [3]:
# Initialize your account
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_aer import AerSimulator
from qiskit.circuit.library import RealAmplitudes, EfficientSU2

service = QiskitRuntimeService(
    channel="ibm_quantum",
    instance="ibm-q-skku/skku/skku-students",
    token="06f802eeac992c43c4106753d7c5fc31414a13eb7d91c99d67fb49cc50569e5b4b011f20c71b9607a07cb43ecc7044557c699022db91d1685ba4c886d5886854",
)
backend = service.backend(directory)


trotterized_ising_circuit, h_params, J_params = create_trotterized_ising_model_circuit(
    8, trotter_steps, time
)
ansatz = EfficientSU2(8, reps=4)

In [4]:
from functions.Train_ZNE import ZNE_stretch

config = {
    "circ": trotterized_ising_circuit,
    "H": hamiltonian,
    "backend": backend,
    "x_amp_dict": X_dict,
    "error_stretch": 1.0,
    "ecr_stretch": 1.0,
    "l": 0,
    "ZNE_factor": [1, 1.1, 1.2],
}
backend_osaka = service.backend(directory)
zne_osaka = ZNE_stretch(**config)
# zne_osaka.load_data('be24d35b-8b87-4277-a288-7ef6f2f15ede')
# zne_osaka.make_data()

In [5]:
import numpy as np
from qiskit_experiments.library import StandardRB, InterleavedRB
from qiskit_experiments.framework import ParallelExperiment, BatchExperiment
import qiskit.circuit.library as circuits
from qiskit_experiments.library.randomized_benchmarking.standard_rb import StandardRB_error

backend_ecr = zne_osaka.ZNE_pulse_ecr(1)
# For simulation
from qiskit_aer import AerSimulator

lengths_2_qubit = np.arange(1, 200, 30)
num_samples = 10
seed = 1010
qubits = X_dict.keys()
data_list_ecr_1 = []
# Run a 1-qubit RB experiment on qubits 1, 2 to determine the error-per-gate of 1-qubit gates
for qubit in qubits:
    exp = StandardRB_error(qubit, lengths_2_qubit, num_samples=num_samples, seed=seed)
    expdata_2q_1_2 = exp.run(backend_ecr).block_for_results()
    data_list_ecr_1.append(expdata_2q_1_2)

In [6]:
import numpy as np
from qiskit_experiments.library import StandardRB, InterleavedRB
from qiskit_experiments.framework import ParallelExperiment, BatchExperiment
import qiskit.circuit.library as circuits


backend_ecr = zne_osaka.ZNE_pulse_ecr(1.2)
# For simulation
from qiskit_aer import AerSimulator

lengths_2_qubit = np.arange(1, 200, 30)
num_samples = 10
seed = 1010
qubits = X_dict.keys()
data_list_ecr_1_2 = []
# Run a 1-qubit RB experiment on qubits 1, 2 to determine the error-per-gate of 1-qubit gates
for qubit in qubits:
    exp = StandardRB_error(qubit, lengths_2_qubit, num_samples=num_samples, seed=seed)
    expdata_2q_1_2 = exp.run(backend_ecr).block_for_results()
    data_list_ecr_1_2.append(expdata_2q_1_2)

In [ ]:
import numpy as np
from qiskit_experiments.library import StandardRB, InterleavedRB
from qiskit_experiments.framework import ParallelExperiment, BatchExperiment
import qiskit.circuit.library as circuits


backend_ecr = zne_osaka.ZNE_pulse_error(1)
# For simulation
from qiskit_aer import AerSimulator

lengths_2_qubit = np.arange(1, 200, 30)
num_samples = 10
seed = 1010
qubits = X_dict.keys()
data_list_error_1 = []
# Run a 1-qubit RB experiment on qubits 1, 2 to determine the error-per-gate of 1-qubit gates
for qubit in qubits:
    exp = StandardRB_error(qubit, lengths_2_qubit, num_samples=num_samples, seed=seed)
    expdata_2q_1_2 = exp.run(backend_ecr).block_for_results()
    data_list_error_1.append(expdata_2q_1_2)

In [ ]:
import numpy as np
from qiskit_experiments.library import StandardRB, InterleavedRB
from qiskit_experiments.framework import ParallelExperiment, BatchExperiment
import qiskit.circuit.library as circuits
from qiskit_experiments.library.randomized_benchmarking.standard_rb import StandardRB_error

backend_ecr = zne_osaka.ZNE_pulse_error(1.2)
# For simulation
from qiskit_aer import AerSimulator

lengths_2_qubit = np.arange(1, 200, 30)
num_samples = 10
seed = 1010
qubits = X_dict.keys()
data_list_error_1_2 = []
# Run a 1-qubit RB experiment on qubits 1, 2 to determine the error-per-gate of 1-qubit gates
for qubit in qubits:
    exp = StandardRB_error(qubit, lengths_2_qubit, num_samples=num_samples, seed=seed)
    expdata_2q_1_2 = exp.run(backend_ecr).block_for_results()
    data_list_error_1_2.append(expdata_2q_1_2)

In [ ]:
import numpy as np
from qiskit_experiments.library import StandardRB, InterleavedRB
from qiskit_experiments.framework import ParallelExperiment, BatchExperiment
import qiskit.circuit.library as circuits
from qiskit_experiments.library.randomized_benchmarking.standard_rb import StandardRB_error

backend_ecr = zne_osaka.ZNE_pulse_ecr(1)
# For simulation
from qiskit_aer import AerSimulator

lengths_2_qubit = np.arange(1, 200, 30)
num_samples = 10
seed = 1010
qubits = X_dict.keys()
data_list_ecr_1 = []
# Run a 1-qubit RB experiment on qubits 1, 2 to determine the error-per-gate of 1-qubit gates
for qubit in qubits:
    exp = StandardRB_error(qubit, lengths_2_qubit, num_samples=num_samples, seed=seed)
    expdata_2q_1_2 = exp.run(backend_ecr).block_for_results()
    data_list_ecr_1.append(expdata_2q_1_2)

In [ ]:
def remove_ecr_gates(circuit):
    new_circuit = QuantumCircuit(circuit.num_qubits)
    for instr, qargs, cargs in circuit.data:

        if instr.name == "measure":
            continue

        elif instr.name == "cx" and qargs[0]._index == 0:
            new_circuit.x(1)

        elif instr.name != "cx":
            new_circuit.append(instr, qargs, cargs)
    return new_circuit

In [ ]:
def get_forward(circuit):
    new_circuit = QuantumCircuit(circuit.num_qubits)
    index = 0
    for instr, qargs, cargs in circuit.data:
        if instr.name == "barrier":
            index += 1
            continue
    index_forward = 0
    for instr, qargs, cargs in circuit.data:
        if (index - 1) == index_forward:
            break
        new_circuit.append(instr, qargs, cargs)
        if instr.name == "barrier":
            index_forward += 1
    return new_circuit


def get_forward_inverse(circuit):
    circuit_forward = get_forward(circuit)
    circuit_reverse = circuit_forward.inverse()
    circuit_forward.append(circuit_reverse, [0, 1])
    return circuit_forward

In [ ]:
import qiskit.circuit.library as circuits
from qiskit.primitives import Sampler

# from functions.Train_ZNE import remove_ecr_gates


exp = StandardRB_error(physical_qubits=(0, 1), lengths=[5], num_samples=5, seed=52)
c = exp.circuits()[2]

sampler = Sampler()
#c = get_forward_inverse(c)
#c.measure_all()
job = sampler.run(c)
job.result()

SamplerResult(quasi_dists=[{0: 0.999999999999992}], metadata=[{}])

In [ ]:
from qiskit import transpile
from qiskit import QuantumCircuit

c = remove_ecr_gates(c)
c = transpile(c, basis_gates=["y", "x", "h", "s", "z", "cx"])



c.measure_all()

In [ ]:
sampler = Sampler()
job = sampler.run(c)
job.result()

SamplerResult(quasi_dists=[{0: 0.999999999999994}], metadata=[{}])